1. Used all columns
2. Scaling
3. Balanced the data
4. Forward selection step with logistic regression

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_excel(r"D:\CRG\Diageo\For Python.xlsx")

df['Quality'] = df['Quality'].map({'Yes': 1, 'No': 0})

In [3]:
df.columns

Index(['Mash B#', 'Malt Hot Water Extract 6-Row (pct w by w)',
       'Malt Predicted Spirit Yield 6-Row (LA by T',
       'Malt Grist Ratio - Coarse (pct w by w)',
       'Malt Grist Ratio - Medium (pct w by w)',
       'Malt Grist Ratio - Fine (pct w by w)', 'Malt Moisture (pct)',
       'Malt Friability 6-Row', 'Water Sensory', 'Water pH', 'Water TDS',
       'Water Hardness', 'Water Chloride', 'Water Alkalinity',
       'Grist Quantity', 'Sparging Temperature 1', 'Sparging Temperature 2',
       'Mash Vessel Turnaround Time', 'Weak Wort Temperature',
       'Raking Arms Speed', 'Mashing - Striking Temp.', 'Draff Loss',
       'Resting Time', 'Wort Collection - Flow Rate',
       'First Wort Collection - Turbidity', 'First Wort Collection - Gravity',
       'Ferm. Yeast Addition', 'CIP Caustic Strength pct',
       'CIP Solution Temperature', 'CIP Circulation time',
       'CIP FV Contact Time', 'Yeast Culture Vessel-Wort Initial Gravity',
       'Yeast Culture Vessel-Wort Final Gra

In [4]:
X = df.drop(columns=['Quality', 'Quality2','Mash B#', 'Water Sensory', 'Water pH', 'Water TDS','Water Hardness', 'Water Chloride', 'Water Alkalinity'])
y = df['Quality']

In [5]:
X

,Malt Hot Water Extract 6-Row (pct w by w),Malt Predicted Spirit Yield 6-Row (LA by T,Malt Grist Ratio - Coarse (pct w by w),Malt Grist Ratio - Medium (pct w by w),Malt Grist Ratio - Fine (pct w by w),Malt Moisture (pct),Malt Friability 6-Row,Grist Quantity,Sparging Temperature 1,Sparging Temperature 2,Mash Vessel Turnaround Time,Weak Wort Temperature,Raking Arms Speed,Mashing - Striking Temp.,Draff Loss,Resting Time,Wort Collection - Flow Rate,First Wort Collection - Turbidity,First Wort Collection - Gravity,Ferm. Yeast Addition,CIP Caustic Strength pct,CIP Solution Temperature,CIP Circulation time,CIP FV Contact Time,Yeast Culture Vessel-Wort Initial Gravity,Yeast Culture Vessel-Wort Final Gravity,Yeast count Million cells per ml,Yeast Culture Vessel Cells pct Viability,Fermentation Volume,Ferm. Set-up Temp,Ferm. Set-up Gravity,Ferm. Set-up pH,Ferm. Max. Temp.,Fermentation Time,Total Fermentation Time,Ferm. Wash Final Gravity,Ferm. wash pH,Ferm. Alcohol(pct) 6-Row,Ferm Wash Residual Sugar,WD 1 - Distillation Time,WD - Alcohol pct in Low Wine,WD - Volume Low Wine,SD - Heads Cut Time,SD - Heart Cut Time,SD - Tail Cut Time,SD - Total Distillation Time,SD - Recovery 6-Row,SD - Average Proof FMS
0,74.023745,341.478997,17,76,7,4.67,69.22,8000,78.3,85.4,275,75.7,0.33,65.4,0.40,25,140,255,1.075,3500,2.0,70,20,20,1.065,1.052,260,99,33986,19,1.060,5.26,34,50,68,1,3.48,8.3,0.41,10.00,20.97038,13127,25,210,270,10.00,584.022000,112.8
1,74.036378,341.537275,17,75,8,4.58,70.90,8000,78.5,85.5,280,75.4,0.33,65.8,0.41,25,140,268,1.072,3500,2.2,70,20,20,1.063,1.051,265,99,33988,19,1.060,5.28,33,52,70,1,3.47,8.3,0.40,10.00,20.85610,13138,25,210,240,9.30,584.539750,112.9
2,74.059020,341.641724,17,75,8,4.60,68.88,8000,78.2,85.3,295,76.6,0.33,65.2,0.38,25,140,265,1.072,3500,2.0,70,20,20,1.062,1.052,260,99,33981,18,1.060,5.31,34,52,70,1,3.50,8.3,0.43,10.00,20.91324,13155,25,210,240,9.30,583.645000,113.0
3,74.012442,341.426855,18,75,7,4.66,69.99,8000,78.4,85.3,295,76.6,0.33,66.1,0.39,25,140,270,1.070,3500,2.2,71,20,20,1.059,1.044,255,99,33986,19,1.060,5.28,34,50,72,1,3.49,8.3,0.41,10.00,20.85610,13143,25,210,240,9.30,584.351250,113.0
4,74.515776,343.748786,17,76,7,4.76,70.10,8000,78.5,85.4,285,76.5,0.33,65.5,0.40,25,140,240,1.075,3500,2.0,70,20,20,1.062,1.052,270,99,33985,18,1.060,5.29,33,52,72,1,3.52,8.3,0.48,10.00,20.91324,13126,25,210,250,9.40,583.599000,112.8
5,74.579340,344.042013,18,76,6,4.63,70.21,8000,78.4,85.3,290,76.4,0.33,65.0,0.41,25,140,265,1.071,3500,2.1,70,20,20,1.065,1.054,260,99,33988,19,1.060,5.30,34,54,70,1,3.42,8.3,0.46,10.00,20.97038,13118,25,210,235,9.25,583.786000,112.7
6,74.049859,341.599464,18,75,7,4.71,70.30,8000,78.4,85.4,275,76.5,0.33,65.2,0.38,25,140,270,1.069,3500,2.0,70,20,20,1.063,1.050,270,99,33974,19,1.059,5.25,34,50,72,1,3.49,8.3,0.48,9.30,20.85610,13146,25,210,240,9.30,584.680875,112.9
7,74.061877,341.654904,17,76,7,4.67,69.99,8000,78.6,85.6,300,76.4,0.33,65.1,0.40,25,140,240,1.072,3500,2.2,70,20,20,1.064,1.054,255,99,33983,19,1.060,5.33,34,52,72,1,3.48,8.2,0.44,10.00,20.68468,13139,25,210,240,9.30,585.009750,113.1
8,74.390000,343.168569,18,75,7,4.53,69.28,8000,78.5,85.4,300,76.4,0.33,65.1,0.43,25,140,233,1.070,3500,2.0,70,20,20,1.063,1.050,270,99,33988,18,1.060,5.29,34,52,70,1,3.42,8.3,0.44,10.00,20.85610,13140,25,210,240,9.30,585.009000,112.8
9,74.334346,342.911831,17,75,8,4.43,68.20,8000,78.4,85.4,280,76.3,0.33,65.3,0.45,25,140,250,1.073,3500,2.1,70,20,20,1.065,1.052,265,99,33985,19,1.058,5.28,34,56,72,1,3.46,8.3,0.39,10.00,20.91324,13129,25,210,270,10.00,586.603500,112.7


In [6]:
y

0     0
1     1
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    1
11    1
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    1
20    0
21    0
22    0
23    0
24    0
25    0
26    0
27    0
28    0
29    0
30    1
31    0
32    0
33    0
34    0
35    0
36    0
37    0
38    0
39    0
40    0
Name: Quality, dtype: int64

In [7]:
print(X.isnull().sum())
print(y.isnull().sum())

Malt Hot Water Extract 6-Row (pct w by w)     0
Malt Predicted Spirit Yield 6-Row (LA by T    0
Malt Grist Ratio - Coarse (pct w by w)        0
Malt Grist Ratio - Medium (pct w by w)        0
Malt Grist Ratio - Fine (pct w by w)          0
Malt Moisture (pct)                           0
Malt Friability 6-Row                         0
Grist Quantity                                0
Sparging Temperature 1                        0
Sparging Temperature 2                        0
Mash Vessel Turnaround Time                   0
Weak Wort Temperature                         0
Raking Arms Speed                             0
Mashing - Striking Temp.                      0
Draff Loss                                    0
Resting Time                                  0
Wort Collection - Flow Rate                   0
First Wort Collection - Turbidity             0
First Wort Collection - Gravity               0
Ferm. Yeast Addition                          0
CIP Caustic Strength pct                

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [9]:
X_scaled

,Malt Hot Water Extract 6-Row (pct w by w),Malt Predicted Spirit Yield 6-Row (LA by T,Malt Grist Ratio - Coarse (pct w by w),Malt Grist Ratio - Medium (pct w by w),Malt Grist Ratio - Fine (pct w by w),Malt Moisture (pct),Malt Friability 6-Row,Grist Quantity,Sparging Temperature 1,Sparging Temperature 2,Mash Vessel Turnaround Time,Weak Wort Temperature,Raking Arms Speed,Mashing - Striking Temp.,Draff Loss,Resting Time,Wort Collection - Flow Rate,First Wort Collection - Turbidity,First Wort Collection - Gravity,Ferm. Yeast Addition,CIP Caustic Strength pct,CIP Solution Temperature,CIP Circulation time,CIP FV Contact Time,Yeast Culture Vessel-Wort Initial Gravity,Yeast Culture Vessel-Wort Final Gravity,Yeast count Million cells per ml,Yeast Culture Vessel Cells pct Viability,Fermentation Volume,Ferm. Set-up Temp,Ferm. Set-up Gravity,Ferm. Set-up pH,Ferm. Max. Temp.,Fermentation Time,Total Fermentation Time,Ferm. Wash Final Gravity,Ferm. wash pH,Ferm. Alcohol(pct) 6-Row,Ferm Wash Residual Sugar,WD 1 - Distillation Time,WD - Alcohol pct in Low Wine,WD - Volume Low Wine,SD - Heads Cut Time,SD - Heart Cut Time,SD - Tail Cut Time,SD - Total Distillation Time,SD - Recovery 6-Row,SD - Average Proof FMS
0,-1.121540,-1.121540,-0.975900,0.884652,0.066741,0.367251,-0.989026,0.0,-1.807513,-0.391659,-1.690808,-0.163625,0.0,0.487815,-0.310052,0.0,0.0,-0.464946,1.492834,0.0,-0.927634,-0.328798,0.0,0.0,1.036535,0.365389,-0.635009,0.0,0.232008,0.605530,0.458333,-0.990314,0.492366,-1.141845,-2.053025,0.0,-0.131684,0.372678,-0.756413,-0.092901,1.118151,-0.668124,0.0,0.0,1.836904,2.016679,-1.387489,-7.764929e-01
1,-1.053409,-1.053409,-0.975900,-1.130388,1.434929,-0.377340,1.308583,0.0,0.404667,0.552931,-1.120716,-0.166428,0.0,1.969326,0.178876,0.0,0.0,0.074959,-0.063256,0.0,1.449428,-0.328798,0.0,0.0,-0.081832,-0.117868,0.088196,0.0,0.791555,0.605530,0.458333,-0.271679,-2.031010,0.123443,-0.695379,0.0,-0.166742,0.372678,-1.057509,-0.092901,-0.210666,-0.005874,0.0,0.0,-0.492366,-0.488892,-0.650367,1.103463e-13
2,-0.931299,-0.931299,-0.975900,-1.130388,1.434929,-0.211875,-1.454018,0.0,-2.913603,-1.336249,0.589558,-0.155214,0.0,-0.252941,-1.287906,0.0,0.0,-0.049635,-0.063256,0.0,-0.927634,-0.328798,0.0,0.0,-0.641015,0.365389,-0.635009,0.0,-1.166862,-1.651446,0.458333,0.806273,0.492366,0.123443,-0.695379,0.0,-0.061566,0.372678,-0.154220,-0.092901,0.453743,1.017605,0.0,0.0,-0.492366,-0.488892,-1.924225,7.764929e-01
3,-1.182498,-1.182498,1.024695,-1.130388,0.066741,0.284518,0.064045,0.0,-0.701423,-1.336249,0.589558,-0.155214,0.0,3.080459,-0.798979,0.0,0.0,0.158021,-1.100649,0.0,1.449428,3.041381,0.0,0.0,-2.318565,-3.500665,-1.358213,0.0,0.232008,0.605530,0.458333,-0.271679,0.492366,-1.141845,0.662266,0.0,-0.096625,0.372678,-0.756413,-0.092901,-0.210666,0.295149,0.0,0.0,-0.492366,-0.488892,-0.918735,7.764929e-01
4,1.532030,1.532030,-0.975900,0.884652,0.066741,1.111841,0.214484,0.0,0.404667,-0.391659,-0.550625,-0.156149,0.0,0.858192,-0.310052,0.0,0.0,-1.087912,1.492834,0.0,-0.927634,-0.328798,0.0,0.0,-0.641015,0.365389,0.811400,0.0,-0.047766,-1.651446,0.458333,0.087638,-2.031010,0.123443,0.662266,0.0,0.008551,0.372678,1.351262,-0.092901,0.453743,-0.728329,0.0,0.0,0.284057,-0.130953,-1.989715,-7.764929e-01
5,1.874837,1.874837,1.024695,0.884652,-1.301447,0.036321,0.364922,0.0,-0.701423,-1.336249,0.019467,-0.157083,0.0,-0.993696,0.178876,0.0,0.0,-0.049635,-0.581952,0.0,0.260897,-0.328798,0.0,0.0,1.036535,1.331903,-0.635009,0.0,0.791555,0.605530,0.458333,0.446956,0.492366,1.388730,-0.695379,0.0,-0.342035,0.372678,0.749069,-0.092901,1.118151,-1.209966,0.0,0.0,-0.880578,-0.667861,-1.723483,-1.552986e+00
6,-0.980705,-0.980705,1.024695,-1.130388,0.066741,0.698180,0.488008,0.0,-0.701423,-0.391659,-1.690808,-0.156149,0.0,-0.252941,-1.287906,0.0,0.0,0.158021,-1.619345,0.0,-0.927634,-0.328798,0.0,0.0,-0.081832,-0.601124,0.811400,0.0,-3.125279,0.605530,-1.250000,-1.349632,0.492366,-1.141845,0.662266,0.0,-0.096625,0.372678,1.351262,-2.899487,-0.210666,0.475763,0.0,0.0,-0

In [10]:
class_distribution = y.value_counts()
print("Class Distribution:\n", class_distribution)

Class Distribution:
 Quality
0    36
1     5
Name: count, dtype: int64


In [11]:
smote = SMOTE(sampling_strategy='auto', k_neighbors=3, random_state=42)  # Adjust k_neighbors
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

In [12]:
X_resampled

,Malt Hot Water Extract 6-Row (pct w by w),Malt Predicted Spirit Yield 6-Row (LA by T,Malt Grist Ratio - Coarse (pct w by w),Malt Grist Ratio - Medium (pct w by w),Malt Grist Ratio - Fine (pct w by w),Malt Moisture (pct),Malt Friability 6-Row,Grist Quantity,Sparging Temperature 1,Sparging Temperature 2,Mash Vessel Turnaround Time,Weak Wort Temperature,Raking Arms Speed,Mashing - Striking Temp.,Draff Loss,Resting Time,Wort Collection - Flow Rate,First Wort Collection - Turbidity,First Wort Collection - Gravity,Ferm. Yeast Addition,CIP Caustic Strength pct,CIP Solution Temperature,CIP Circulation time,CIP FV Contact Time,Yeast Culture Vessel-Wort Initial Gravity,Yeast Culture Vessel-Wort Final Gravity,Yeast count Million cells per ml,Yeast Culture Vessel Cells pct Viability,Fermentation Volume,Ferm. Set-up Temp,Ferm. Set-up Gravity,Ferm. Set-up pH,Ferm. Max. Temp.,Fermentation Time,Total Fermentation Time,Ferm. Wash Final Gravity,Ferm. wash pH,Ferm. Alcohol(pct) 6-Row,Ferm Wash Residual Sugar,WD 1 - Distillation Time,WD - Alcohol pct in Low Wine,WD - Volume Low Wine,SD - Heads Cut Time,SD - Heart Cut Time,SD - Tail Cut Time,SD - Total Distillation Time,SD - Recovery 6-Row,SD - Average Proof FMS
0,-1.121540,-1.121540,-0.975900,0.884652,0.066741,0.367251,-0.989026,0.0,-1.807513,-0.391659,-1.690808,-0.163625,0.0,0.487815,-0.310052,0.0,0.0,-0.464946,1.492834,0.0,-0.927634,-0.328798,0.0,0.0,1.036535,0.365389,-0.635009,0.0,0.232008,0.605530,0.458333,-0.990314,0.492366,-1.141845,-2.053025,0.0,-0.131684,0.372678,-0.756413,-0.092901,1.118151,-0.668124,0.0,0.0,1.836904,2.016679,-1.387489,-7.764929e-01
1,-1.053409,-1.053409,-0.975900,-1.130388,1.434929,-0.377340,1.308583,0.0,0.404667,0.552931,-1.120716,-0.166428,0.0,1.969326,0.178876,0.0,0.0,0.074959,-0.063256,0.0,1.449428,-0.328798,0.0,0.0,-0.081832,-0.117868,0.088196,0.0,0.791555,0.605530,0.458333,-0.271679,-2.031010,0.123443,-0.695379,0.0,-0.166742,0.372678,-1.057509,-0.092901,-0.210666,-0.005874,0.0,0.0,-0.492366,-0.488892,-0.650367,1.103463e-13
2,-0.931299,-0.931299,-0.975900,-1.130388,1.434929,-0.211875,-1.454018,0.0,-2.913603,-1.336249,0.589558,-0.155214,0.0,-0.252941,-1.287906,0.0,0.0,-0.049635,-0.063256,0.0,-0.927634,-0.328798,0.0,0.0,-0.641015,0.365389,-0.635009,0.0,-1.166862,-1.651446,0.458333,0.806273,0.492366,0.123443,-0.695379,0.0,-0.061566,0.372678,-0.154220,-0.092901,0.453743,1.017605,0.0,0.0,-0.492366,-0.488892,-1.924225,7.764929e-01
3,-1.182498,-1.182498,1.024695,-1.130388,0.066741,0.284518,0.064045,0.0,-0.701423,-1.336249,0.589558,-0.155214,0.0,3.080459,-0.798979,0.0,0.0,0.158021,-1.100649,0.0,1.449428,3.041381,0.0,0.0,-2.318565,-3.500665,-1.358213,0.0,0.232008,0.605530,0.458333,-0.271679,0.492366,-1.141845,0.662266,0.0,-0.096625,0.372678,-0.756413,-0.092901,-0.210666,0.295149,0.0,0.0,-0.492366,-0.488892,-0.918735,7.764929e-01
4,1.532030,1.532030,-0.975900,0.884652,0.066741,1.111841,0.214484,0.0,0.404667,-0.391659,-0.550625,-0.156149,0.0,0.858192,-0.310052,0.0,0.0,-1.087912,1.492834,0.0,-0.927634,-0.328798,0.0,0.0,-0.641015,0.365389,0.811400,0.0,-0.047766,-1.651446,0.458333,0.087638,-2.031010,0.123443,0.662266,0.0,0.008551,0.372678,1.351262,-0.092901,0.453743,-0.728329,0.0,0.0,0.284057,-0.130953,-1.989715,-7.764929e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,1.674465,1.674465,0.955898,0.815357,-1.207347,-1.735423,1.453848,0.0,-0.663386,0.552931,-1.120716,-0.156502,0.0,0.896403,-0.765352,0.0,0.0,-1.047923,-0.564115,0.0,-0.845890,-0.328798,0.0,0.0,0.998076,0.348771,-0.610139,0.0,0.251250,0.605530,0.458333,-1.312562,0.405591,-1.098333,-0.695379,0.0,-0.369860,0.372678,0.977686,-0.092901,1.072455,-3.377661,0.0,0.0,-0.492366,-0.488892,-0.392946,-7.497904e-01
68,0.600527,0.600527,0.843282,-1.130388,0.190808,-1.054411,0.363440,0.0,1.410457,1.411866,0.952867,-0.156231,0.0,-0.725011,0.178876,0.0,0.0,-0.038337,-0.063256,0.0,0

In [13]:
y_resampled

0     0
1     1
2     0
3     0
4     0
     ..
67    1
68    1
69    1
70    1
71    1
Name: Quality, Length: 72, dtype: int64

In [14]:
y_resampled.value_counts()

Quality
0    36
1    36
Name: count, dtype: int64

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)

In [16]:
selected_variables = []
remaining_variables = list(X_train.columns)
metrics_history = []

logreg = LogisticRegression(max_iter=10000, solver='liblinear')

def calculate_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # Probability scores for AUC
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)
    return accuracy, precision, recall, f1, auc


step = 0
improvement = True

# Forward selection + backward elimination
"""while remaining_variables and improvement:
    step += 1
    improvement = False
    best_accuracy = 0
    best_variable = None
    best_metrics = None
    
    # Forward Selection: Try adding each remaining variable
    for variable in remaining_variables:
        trial_variables = selected_variables + [variable]
        X_train_subset = X_train[trial_variables]
        X_test_subset = X_test[trial_variables]
        
        # Fit Logistic Regression model
        logreg.fit(X_train_subset, y_train)
        
        # Calculate metrics
        accuracy, precision, recall, f1, auc = calculate_metrics(logreg, X_test_subset, y_test)
        
        # Select the variable that gives the highest accuracy
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_variable = variable
            best_metrics = (accuracy, precision, recall, f1, auc)

    # If a variable improves the model, add it
    if best_variable is not None:
        selected_variables.append(best_variable)
        remaining_variables.remove(best_variable)
        metrics_history.append({
            'Step': step,
            'Action': f'Added {best_variable}',
            'Variables Selected': ', '.join(selected_variables),
            'Accuracy': round(best_metrics[0] * 100, 2),
            'Precision': round(best_metrics[1] * 100, 2),
            'Recall': round(best_metrics[2] * 100, 2),
            'F1-Score': round(best_metrics[3] * 100, 2),
            'AUC': round(best_metrics[4], 2)
        })
        print(f"Step {step}: Added '{best_variable}'")
        print(f"Metrics -> Accuracy: {best_metrics[0]:.4f}, Precision: {best_metrics[1]:.4f}, Recall: {best_metrics[2]:.4f}, F1-Score: {best_metrics[3]:.4f}, AUC: {best_metrics[4]:.4f}\n")
        improvement = True

    # Backward Elimination: Check if any selected variable should be removed
    if len(selected_variables) > 1:
        worst_accuracy = best_accuracy
        worst_variable = None
        
        for variable in selected_variables:
            trial_variables = [v for v in selected_variables if v != variable]
            X_train_subset = X_train[trial_variables]
            X_test_subset = X_test[trial_variables]
            
            # Fit Logistic Regression model
            logreg.fit(X_train_subset, y_train)
            
            # Calculate metrics
            accuracy, _, _, _, _ = calculate_metrics(logreg, X_test_subset, y_test)
            
            # Check if removing the variable improves accuracy
            if accuracy >= worst_accuracy:
                worst_accuracy = accuracy
                worst_variable = variable

        # If removing a variable improves or maintains accuracy, remove it
        if worst_variable is not None:
            selected_variables.remove(worst_variable)
            remaining_variables.append(worst_variable)
            metrics_history.append({
                'Step': step,
                'Action': f'Removed {worst_variable}',
                'Variables Selected': ', '.join(selected_variables),
                'Accuracy': round(worst_accuracy * 100, 2),
                'Precision': round(best_metrics[1] * 100, 2),  # Using last best metrics
                'Recall': round(best_metrics[2] * 100, 2),
                'F1-Score': round(best_metrics[3] * 100, 2),
                'AUC': round(best_metrics[4], 2)
            })
            print(f"Step {step}: Removed '{worst_variable}'")
            print(f"Metrics -> Accuracy: {worst_accuracy:.4f}\n")
            improvement = True"""

'while remaining_variables and improvement:\n    step += 1\n    improvement = False\n    best_accuracy = 0\n    best_variable = None\n    best_metrics = None\n    \n    # Forward Selection: Try adding each remaining variable\n    for variable in remaining_variables:\n        trial_variables = selected_variables + [variable]\n        X_train_subset = X_train[trial_variables]\n        X_test_subset = X_test[trial_variables]\n        \n        # Fit Logistic Regression model\n        logreg.fit(X_train_subset, y_train)\n        \n        # Calculate metrics\n        accuracy, precision, recall, f1, auc = calculate_metrics(logreg, X_test_subset, y_test)\n        \n        # Select the variable that gives the highest accuracy\n        if accuracy > best_accuracy:\n            best_accuracy = accuracy\n            best_variable = variable\n            best_metrics = (accuracy, precision, recall, f1, auc)\n\n    # If a variable improves the model, add it\n    if best_variable is not None:\n  

In [17]:
# Forward Selection

while remaining_variables and improvement:
    step += 1
    improvement = False
    best_accuracy = 0
    best_variable = None
    best_metrics = None
    
    # Forward Selection: Try adding each remaining variable
    for variable in remaining_variables:
        trial_variables = selected_variables + [variable]
        X_train_subset = X_train[trial_variables]
        X_test_subset = X_test[trial_variables]
        
        # Fit Logistic Regression model
        logreg.fit(X_train_subset, y_train)
        
        # Calculate metrics
        accuracy, precision, recall, f1, auc = calculate_metrics(logreg, X_test_subset, y_test)
        
        # Select the variable that gives the highest accuracy
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_variable = variable
            best_metrics = (accuracy, precision, recall, f1, auc)

    # If a variable improves the model, add it
    if best_variable is not None:
        selected_variables.append(best_variable)
        remaining_variables.remove(best_variable)
        metrics_history.append({
            'Step': step,
            'Action': f'Added {best_variable}',
            'Variables Selected': ', '.join(selected_variables),
            'Accuracy': round(best_metrics[0] * 100, 2),
            'Precision': round(best_metrics[1] * 100, 2),
            'Recall': round(best_metrics[2] * 100, 2),
            'F1-Score': round(best_metrics[3] * 100, 2),
            'AUC': round(best_metrics[4], 2)
        })
        print(f"Step {step}: Added '{best_variable}'")
        print(f"Metrics -> Accuracy: {best_metrics[0]:.4f}, Precision: {best_metrics[1]:.4f}, Recall: {best_metrics[2]:.4f}, F1-Score: {best_metrics[3]:.4f}, AUC: {best_metrics[4]:.4f}\n")
        improvement = True

Step 1: Added 'Malt Friability 6-Row'
Metrics -> Accuracy: 0.8182, Precision: 0.7778, Recall: 0.7778, F1-Score: 0.7778, AUC: 0.8889

Step 2: Added 'Yeast Culture Vessel-Wort Initial Gravity'
Metrics -> Accuracy: 0.9545, Precision: 0.9000, Recall: 1.0000, F1-Score: 0.9474, AUC: 0.9231

Step 3: Added 'CIP Solution Temperature'
Metrics -> Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000, AUC: 1.0000

Step 4: Added 'Malt Grist Ratio - Coarse (pct w by w)'
Metrics -> Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000, AUC: 1.0000

Step 5: Added 'Grist Quantity'
Metrics -> Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000, AUC: 1.0000

Step 6: Added 'Raking Arms Speed'
Metrics -> Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000, AUC: 1.0000

Step 7: Added 'Resting Time'
Metrics -> Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000, AUC: 1.0000

Step 8: Added 'Wort Collection - Flow Rate'
Metri

In [18]:
pd.set_option('display.max_rows', None)
results_df = pd.DataFrame(metrics_history)
print("Stepwise Logistic Regression Results:")
results_df

Stepwise Logistic Regression Results:


,Step,Action,Variables Selected,Accuracy,Precision,Recall,F1-Score,AUC
0,1,Added Malt Friability 6-Row,Malt Friability 6-Row,81.82,77.78,77.78,77.78,0.89
1,2,Added Yeast Culture Vessel-Wort Initial Gravity,"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",95.45,90.00,100.00,94.74,0.92
2,3,Added CIP Solution Temperature,"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
3,4,Added Malt Grist Ratio - Coarse (pct w by w),"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
4,5,Added Grist Quantity,"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
5,6,Added Raking Arms Speed,"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
6,7,Added Resting Time,"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
7,8,Added Wort Collection - Flow Rate,"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
8,9,Added First Wort Collection - Gravity,"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
9,10,Added Malt Grist Ratio - Fine (pct w by w),"Malt Friability 6-Row, Yeast Culture Vessel-Wo...",100.00,100.00,100.00,100.00,1.00
